In [23]:
from Class_PAS_Data_Extract import PASDataEngine
from Class_PAS_Product import Product
from Class_PAS_Email import EmailUpdate
import Class_PAS_Email
import Class_PAS_Graph
import importlib
import pandas as pd

In [2]:
importlib.reload(importlib.import_module('Class_PAS_Product'))

<module 'Class_PAS_Product' from 'd:\\Python\\PAS_Graphs\\Class_PAS_Product.py'>

In [3]:
plotPaths = "plots/"
key_file = r'\\shuser-Prod.intel.com\shProduser$\dagarcia\keys\secret.key'
credential_file = r'\\shuser-Prod.intel.com\shProduser$\dagarcia\EncryptedCredentials\credentials.encrypted'


Server = "smtpauth.intel.com"
Port = "587"
UserEmail = "david.a.garcia@intel.com"


In [4]:
def read_excel_to_dataframe(file_path, sheet_name, halt_on_error=True):
    """
    Reads a specific worksheet from an Excel file into a Pandas DataFrame.

    Parameters:
    - file_path (str): The path to the Excel file.
    - sheet_name (str or int): The name or index of the worksheet to load.
    - halt_on_error (bool): Flag indicating whether to halt execution on error.
                            If False, the function will return None on error.

    Returns:
    - DataFrame or None: The DataFrame if successful, or None if an error occurs and halt_on_error is False.
    """
    try:
        # Attempt to read the specified worksheet into a DataFrame
        df = pd.read_excel(file_path, sheet_name=sheet_name)
        return df
    except Exception as e:
        # Handle the error based on the halt_on_error flag
        if halt_on_error:
            print(f"Error reading {file_path}, sheet {sheet_name}: {e}")
            raise  # Re-raise the exception to halt execution
        else:
            print(f"Error reading {file_path}, sheet {sheet_name}: {e}. Returning None.")
            return None



In [5]:
config_file = r'\\azshfs.intel.com\AZAnalysis$\1272_MAODATA\Config\PDE\dagarcia\PAS_CONFIG\P1275_Config.xlsx'

pas_config = read_excel_to_dataframe(config_file, 'PlotConfig', halt_on_error=True)
email_config = read_excel_to_dataframe(config_file, 'EmailConfig', halt_on_error=True)
reticle_config = read_excel_to_dataframe(config_file, 'ReticleConfig', halt_on_error=False)

In [6]:
if  pas_config is not None:
    unique_combos = pas_config.groupby(['PRODUCT', 'FAB_PROD', 'RET_PROD'])['COMMIT'].max().reset_index()

    products = unique_combos.set_index('PRODUCT').to_dict(orient='index')
    
    for product, details in products.items():
        print(f"Processing Product: {product}")
        print(f"FAB_PROD: {details['FAB_PROD']}, RET_PROD: {details['RET_PROD']}, COMMIT: {details['COMMIT']}")



Processing Product: Sundance Mesa 1
FAB_PROD: 8PWQCVA, RET_PROD: 8PWQCA, COMMIT: 2025-07-28 18:00:00


In [7]:
# products['Sundance Mesa 1']
pas_config

,PRODUCT,TITLE,LOT,COMMIT,FAB_PROD,RET_PROD,ORIG_COMMIT
0,Sundance Mesa 1,Lead Lot,L5138740,2025-07-28 18:00:00,8PWQCVA,8PWQCA,2025-08-14
1,Sundance Mesa 1,Scout 1,L5139550,2025-07-27 18:00:00,8PWQCVA,8PWQCA,2025-08-04
2,Sundance Mesa 1,Scout 2,L5139560,2025-07-27 18:00:00,8PWQCVA,8PWQCA,2025-08-04


In [8]:
dataengine = PASDataEngine()

In [9]:

prod = Product('8PWQCA', products['Sundance Mesa 1'],  dataengine, ret_version=reticle_config, lots=None, debug_flag=True)

d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:92: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Product.py:118: FutureWarning: The provided callable <function min at 0x0000019A1E3FCF40> is currently using DataFrameGroupBy.min. In a future version of pandas, the provided callable will be used directly. To keep current behavior pass the string "min" instead.
  RetData = pd.pivot_table(RetData,index=['LAYER'], values=['TAPEIN_TREND','ITO_TREND','FAB_REQUIREDDATE','IMO_TREND','SHIPDATE'], aggfunc=np.min).reset_index()


In [10]:
prod.add_lot('L5138740', 'Lead Lot')
prod.add_lot('L5139550', 'Scout 1')
prod.add_lot('L5139560', 'Scout 2')

d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:41: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string URI or sqlite3 DBAPI2 connection. Other DBAPI2 objects are not tested. Please consider using SQLAlchemy.
  df = pd.read_sql(query, conn)
d:\Python\PAS_Graphs\Class_PAS_Data_Extract.py:114: UserWarning: pandas only supports SQLAlchemy connectable (engine/connection) or database string

In [11]:
prod.build_plot_data()


Difference in days: 158 days
Rounded difference: 161.0 days
ymin_date: 2025-02-22 00:00:00
ymin_val: 0
ymax_date: 2025-08-02 00:00:00
ymax_val: 161.0


In [12]:


importlib.reload(importlib.import_module('Class_PAS_Graph'))
myGraph = Class_PAS_Graph.PASPlot(prod,plotPaths)
myGraph.make_plot()

commit_date: 2025-07-28 18:00:00
trend_date: 2025-07-23 17:13:27.203438368


d:\Python\PAS_Graphs\Class_PAS_Graph.py:106: UserWarning: set_ticklabels() should only be used with a fixed number of ticks, i.e. after set_ticks() or using a FixedLocator.
  ax1.set_xticklabels(plotdata['LAYER'], rotation=90, fontsize=8)


In [13]:
prod.plot_data_raw

,LAYER,EXEC_SEQ,TI,TO,ESD,SHIP,FRD,NPI PLAN,Lead Lot TREND,Lead Lot ACTUAL,Scout 1 TREND,Scout 1 ACTUAL,Scout 2 TREND,Scout 2 ACTUAL
0,START,1.0,NaT,NaT,NaT,NaT,NaT,2025-04-03 01:38:14.607449856,2025-03-24 17:55:06.000000000,2025-03-24 17:55:06,2025-03-24 17:55:07.000000000,2025-03-24 17:55:07,2025-03-24 17:55:06.000000000,2025-03-24 17:55:06
1,DCB,25.0,2025-03-07,2025-03-12,NaT,2025-03-20 05:25:17,2025-03-23,2025-04-03 21:45:07.214899712,2025-04-02 05:31:22.000000000,2025-04-02 05:31:22,2025-03-27 04:50:18.000000000,2025-03-27 04:50:18,2025-03-27 07:23:51.000000000,2025-03-27 07:23:51
2,DN1,81.0,2025-03-07,2025-03-12,NaT,2025-03-19 20:37:57,2025-03-23,2025-04-05 13:58:52.429799426,2025-04-04 21:37:44.000000000,2025-04-04 21:37:44,2025-03-29 17:37:43.000000000,2025-03-29 17:37:43,2025-03-29 13:45:46.000000000,2025-03-29 13:45:46
3,DN2,98.0,2025-03-07,2025-03-12,NaT,2025-03-20 22:30:19,2025-03-23,2025-04-06 02:02:59.994269340,2025-04-05 06:00:07.000000000,2025-04-05 06:00:07,2025-03-30 03:38:11.000000000,2025-03-30 03:38:11,2025-03-30 02:34:44.000000000,2025-03-30 02:34:44
4,DN3,115.0,2025-03-07,2025-03-12,NaT,2025-03-20 02:55:27,2025-03-23,2025-04-06 14:07:07.558739254,2025-04-05 19:40:30.000000000,2025-04-05 19:40:30,2025-03-31 11:23:34.000000000,2025-03-31 11:23:34,2025-03-30 23:15:49.000000000,2025-03-30 23:15:49
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
97,CE3,2458.0,2025-03-07,2025-03-18,NaT,2025-03-27 17:10:54,2025-05-25,2025-07-25 17:10:44.177650428,2025-07-19 12:39:04.166189086,NaT,2025-07-17 20:01:49.991403983,NaT,2025-07-17 23:55:29.991403983,NaT
98,GV0,2473.0,2025-03-07,2025-03-18,NaT,2025-03-28 05:32:47,2025-05-26,2025-07-26 13:17:36.785100286,2025-07-20 08:45:56.773638942,NaT,2025-07-18 17:10:36.065902550,NaT,2025-07-18 21:04:16.065902550,NaT
99,GM1,2498.0,2025-03-07,2025-03-18,NaT,2025-03-28 06:50:08,2025-05-26,2025-07-28 01:29:59.478510028,2025-07-21 20:58:19.467048684,NaT,2025-07-20 07:14:22.999999971,NaT,2025-07-20 11:08:02.999999971,NaT
100,GV1,2522.0,2025-03-07,2025-04-09,NaT,2025-04-21 07:08:34,2025-05-26,2025-07-29 17:43:44.693409742,2025-07-23 13:12:04.681948397,NaT,2025-07-22 01:31:55.148997105,NaT,2025-07-22 05:25:35.148997105,NaT


In [35]:
importlib.reload(importlib.import_module('Class_PAS_Email'))

email = Class_PAS_Email.EmailUpdate(
    server = Server,
    port = Port,
    email = UserEmail,
    key_file=key_file, 
    credential_file=credential_file, 
    email_config=email_config,
    plot_folder=plotPaths,
    subject="Test PAS RUN",
    debug=False)


In [36]:
email.addProduct('8PWQCA', prod.plot_data_raw)
email.html_body

"<html><body><p style='font-weight: bold; text-decoration: underline; font-size: 16px;'>8PWQCA</p><p>Next 10 layer look ahead: Reticles have all shipped!</p><img src='cid:174772471470.14912.12060498644017790619'>"

In [30]:
email.html_body

"<html><body><p style='font-weight: bold; text-decoration: underline; font-size: 16px;'>8PWQCA</p><p>Next 10 layer look ahead: Reticles have all shipped!</p><img src='cid:174772460069.14912.5627425640051946623'>"

In [37]:
email.send_email()